# 消费信贷违约风险特征分析与风险分层｜Python / Pandas

## 项目背景
本项目基于公开消费信贷数据，从历史逾期行为、授信额度、年龄、额度使用和还款行为等维度分析客户下一期违约风险，并在单特征分析基础上构造可解释的风险信号组合与风险分层。

## 分析目标
1. 识别与下一期违约风险关联较明显的客户行为与账户特征；
2. 使用 Pandas 完成数据检查、特征构造、分组统计和风险比较；
3. 将多个风险信号进行组合，观察风险叠加后的区分效果；
4. 形成一套可解释、可复现的消费信贷违约风险分析流程。

## 分析框架
`数据检查 → 单特征风险分析 → 风险特征工程 → 规则命中分析 → 风险分层`

> 本 Notebook 展示 Python / Pandas 分析部分；SQL 分析将在独立 Notebook 中展示。


## 1. 数据读取与字段整理

数据来源为 UCI **Default of Credit Card Clients** 数据集，共包含 30,000 条客户记录。目标变量表示客户下一期是否发生违约。


In [1]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

In [2]:
credit_dataset = fetch_ucirepo(id=350)

X = credit_dataset.data.features.copy()
y = credit_dataset.data.targets.copy()

raw_df = pd.concat([X, y], axis=1)
raw_df.head()

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X15,X16,X17,X18,X19,X20,X21,X22,X23,Y
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


### 1.1 目标变量与业务字段重命名

为便于后续分析，将目标变量统一命名为 `default_flag`：
- `0`：下一期未违约
- `1`：下一期违约

同时将原始 `X1-X23` 字段重命名为更直观的业务字段名，便于后续构造逾期、额度使用和还款相关特征。


In [5]:
target_column = y.columns[0]

raw_df.rename(
    columns={target_column: "default_flag"},
    inplace=True
)

column_mapping = {
    "X1": "credit_limit",
    "X2": "sex",
    "X3": "education",
    "X4": "marriage",
    "X5": "age",
    "X6": "repay_status_sep",
    "X7": "repay_status_aug",
    "X8": "repay_status_jul",
    "X9": "repay_status_jun",
    "X10": "repay_status_may",
    "X11": "repay_status_apr",
    "X12": "bill_amount_sep",
    "X13": "bill_amount_aug",
    "X14": "bill_amount_jul",
    "X15": "bill_amount_jun",
    "X16": "bill_amount_may",
    "X17": "bill_amount_apr",
    "X18": "payment_amount_sep",
    "X19": "payment_amount_aug",
    "X20": "payment_amount_jul",
    "X21": "payment_amount_jun",
    "X22": "payment_amount_may",
    "X23": "payment_amount_apr"
}

raw_df.rename(columns=column_mapping, inplace=True)
raw_df.head()

,credit_limit,sex,education,marriage,age,repay_status_sep,repay_status_aug,repay_status_jul,repay_status_jun,repay_status_may,...,bill_amount_jun,bill_amount_may,bill_amount_apr,payment_amount_sep,payment_amount_aug,payment_amount_jul,payment_amount_jun,payment_amount_may,payment_amount_apr,default_flag
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


## 2. 数据质量检查与整体违约情况

对数据规模、缺失值、完全重复记录和目标变量分布进行基础检查。由于公开数据中缺少可靠的唯一客户标识，完全重复记录仅进行数量记录，不直接删除。


In [4]:
print("数据形状：", raw_df.shape)
print("缺失值总数：", raw_df.isna().sum().sum())
print("完全重复行数量：", raw_df.duplicated().sum())
print("目标变量列是否存在：", "default_flag" in raw_df.columns)

数据形状： (30000, 24)
缺失值总数： 0
完全重复行数量： 35
目标变量列是否存在： True


In [ ]:
print("违约人数统计：")
print(raw_df["default_flag"].value_counts())

print("\n整体违约率：")
print(f'{raw_df["default_flag"].mean():.2%}')

### 数据概览

数据共 **30,000** 条记录，整体下一期违约率为 **22.12%**。由于 `default_flag` 为 0/1 变量，后续使用其组内平均值表示对应客群的违约率。


In [ ]:
credit_df = raw_df.copy()

repay_status_cols = [
    "repay_status_sep",
    "repay_status_aug",
    "repay_status_jul",
    "repay_status_jun",
    "repay_status_may",
    "repay_status_apr"
]

## 3. 历史逾期行为与违约风险

本部分从逾期状态、逾期严重程度、逾期频率和连续性四个角度分析历史还款表现与下一期违约风险之间的关系。


### 3.1 9月是否逾期

**分析问题：** 最近一期已经出现逾期的客户，下一期违约风险是否明显更高？

定义 `repay_status_sep > 0` 为9月存在逾期，并比较逾期与未逾期客户的下一期违约率。


In [ ]:
sep_overdue_mask = credit_df["repay_status_sep"] > 0

sep_overdue_df = credit_df.loc[sep_overdue_mask]
sep_non_overdue_df = credit_df.loc[~sep_overdue_mask]

sep_overdue_default_rate = sep_overdue_df["default_flag"].mean()
sep_non_overdue_default_rate = sep_non_overdue_df["default_flag"].mean()

sep_overdue_summary = pd.DataFrame({
    "客户类型": ["9月逾期客户", "9月未逾期客户"],
    "客户人数": [len(sep_overdue_df), len(sep_non_overdue_df)],
    "下一期违约率": [sep_overdue_default_rate, sep_non_overdue_default_rate]
})
sep_overdue_summary

In [ ]:
print("逾期客户违约率是未逾期客户的", (sep_overdue_default_rate / sep_non_overdue_default_rate).round(2), "倍")
print("两组违约率相差", ((sep_overdue_default_rate - sep_non_overdue_default_rate) * 100).round(2), "个百分点")

**分析结论：** 9月逾期客户下一期违约率约为 **50.29%**，未逾期客户约为 **13.83%**；前者约为后者的 **3.64 倍**，相差约 **36.46 个百分点**。最近一期逾期状态具有明显的风险区分能力。


### 3.2 9月逾期严重程度

**分析问题：** 逾期程度加重时，下一期违约风险是否同步上升？

将客户划分为未逾期、逾期1个月、逾期2个月、逾期3个月及以上四组，对比各组下一期违约率。


In [ ]:
sep_non_overdue_df = credit_df.loc[credit_df["repay_status_sep"] <= 0]
sep_delay_1_df = credit_df.loc[credit_df["repay_status_sep"] == 1]
sep_delay_2_df = credit_df.loc[credit_df["repay_status_sep"] == 2]
sep_delay_3plus_df = credit_df.loc[credit_df["repay_status_sep"] >= 3]

sep_non_overdue_default_rate = sep_non_overdue_df["default_flag"].mean()
sep_delay_1_default_rate = sep_delay_1_df["default_flag"].mean()
sep_delay_2_default_rate = sep_delay_2_df["default_flag"].mean()
sep_delay_3plus_default_rate = sep_delay_3plus_df["default_flag"].mean()

sep_delay_level_summary = pd.DataFrame({
    "逾期程度": ["未逾期", "逾期1个月", "逾期2个月", "逾期3个月及以上"],
    "客户人数": [len(sep_non_overdue_df), len(sep_delay_1_df), len(sep_delay_2_df), len(sep_delay_3plus_df)],
    "下一期违约率": [sep_non_overdue_default_rate, sep_delay_1_default_rate, sep_delay_2_default_rate, sep_delay_3plus_default_rate]
})
sep_delay_level_summary

**分析结论：** 四组违约率分别约为 **13.83%、33.95%、69.14%、71.92%**。整体来看，逾期程度越严重，下一期违约率越高，逾期严重程度具有较强的风险区分能力。


### 3.3 近6个月逾期频率

**分析问题：** 近6个月出现逾期的月份越多，下一期违约风险是否越高？

将6个月还款状态中 `> 0` 的月份记为一次逾期，并统计每个客户近6个月的逾期月份数。


In [ ]:
credit_df["overdue_month_count_6m"] = (credit_df[repay_status_cols] > 0).sum(axis=1)

overdue_month_count_counts = credit_df["overdue_month_count_6m"].value_counts().sort_index()
overdue_month_count_default_rate = credit_df.groupby("overdue_month_count_6m")["default_flag"].mean()

overdue_month_count_summary = pd.DataFrame({
    "客户人数": overdue_month_count_counts,
    "下一期违约率": overdue_month_count_default_rate
})
overdue_month_count_summary

**分析结论：** 下一期违约率随近6个月逾期月份数增加而整体上升。未出现正向逾期的客户违约率约为 **11.71%**，6个月均出现逾期的客户违约率约为 **70.32%**，说明逾期频率能够有效刻画客户风险差异。


### 3.4 近6个月最大逾期程度

**分析问题：** 客户历史上出现过的最严重逾期状态，是否能够反映后续违约风险？

对近6个月还款状态按行取最大值。由于原始还款状态包含负值，使用 `clip(lower=0)` 将负值调整为0，使该指标表示客户近6个月的最大正向逾期程度。


In [ ]:
credit_df["max_repay_status_6m"] = credit_df[repay_status_cols].max(axis=1)
credit_df["max_overdue_months_6m"] = credit_df["max_repay_status_6m"].clip(lower=0)

max_overdue_months_counts = credit_df["max_overdue_months_6m"].value_counts().sort_index()
max_overdue_default_rate = credit_df.groupby("max_overdue_months_6m")["default_flag"].mean()

max_overdue_summary = pd.DataFrame({
    "客户人数": max_overdue_months_counts,
    "下一期违约率": max_overdue_default_rate
})
max_overdue_summary

**分析结论：** 最大逾期程度从0上升至4个月时，违约率由约 **11.71%** 上升至 **64.22%**。5个月以上客群样本量较小，违约率存在波动，因此高逾期等级需要结合样本量谨慎解释。


### 3.5 连续逾期与违约风险

**分析问题：** 出现连续逾期的客户，是否比没有连续逾期的客户具有更高的后续违约风险？

若任意一组相邻月份同时出现正向逾期，则定义为“近6个月存在连续逾期”。


In [ ]:
credit_df["continuous_overdue_sep_aug"] = (credit_df["repay_status_sep"] > 0) & (credit_df["repay_status_aug"] > 0)
credit_df["continuous_overdue_aug_jul"] = (credit_df["repay_status_aug"] > 0) & (credit_df["repay_status_jul"] > 0)
credit_df["continuous_overdue_jul_jun"] = (credit_df["repay_status_jul"] > 0) & (credit_df["repay_status_jun"] > 0)
credit_df["continuous_overdue_jun_may"] = (credit_df["repay_status_jun"] > 0) & (credit_df["repay_status_may"] > 0)
credit_df["continuous_overdue_may_apr"] = (credit_df["repay_status_may"] > 0) & (credit_df["repay_status_apr"] > 0)

continuous_overdue_cols = [
    "continuous_overdue_sep_aug",
    "continuous_overdue_aug_jul",
    "continuous_overdue_jul_jun",
    "continuous_overdue_jun_may",
    "continuous_overdue_may_apr"
]

credit_df["continuous_overdue_flag_6m"] = credit_df[continuous_overdue_cols].any(axis=1)

In [ ]:
continuous_overdue_counts = credit_df["continuous_overdue_flag_6m"].value_counts().sort_index()
continuous_overdue_default_rate = credit_df.groupby("continuous_overdue_flag_6m")["default_flag"].mean()

continuous_overdue_summary = pd.DataFrame({
    "客户人数": continuous_overdue_counts,
    "下一期违约率": continuous_overdue_default_rate
})
continuous_overdue_summary

In [ ]:
continuous_overdue_rate = continuous_overdue_default_rate[True]
non_continuous_overdue_rate = continuous_overdue_default_rate[False]

print("连续逾期客户违约率是非连续逾期客户的", (continuous_overdue_rate / non_continuous_overdue_rate).round(2), "倍")
print("两组违约率相差", ((continuous_overdue_rate - non_continuous_overdue_rate) * 100).round(2), "个百分点")

**分析结论：** 连续逾期客户下一期违约率约为 **53.53%**，非连续逾期客户约为 **15.49%**，前者约为后者的 **3.46 倍**。连续逾期是本数据中区分度较强的风险信号之一。


## 4. 客户与账户特征分析

在逾期行为之外，进一步分析客户属性和账户使用特征是否能够补充解释违约风险差异。


### 4.1 授信额度与违约风险

**分析问题：** 不同授信额度客群的下一期违约率是否存在明显差异？

先观察授信额度分布，再按额度区间分组比较下一期违约率。


In [ ]:
credit_df["credit_limit"].describe()

In [ ]:
credit_df["credit_limit_group"] = pd.cut(
    credit_df["credit_limit"],
    bins=[0, 50000, 100000, 200000, 300000, float("inf")],
    labels=["低额度", "较低额度", "中等额度", "较高额度", "高额度"],
    right=False
)

credit_limit_group_counts = credit_df["credit_limit_group"].value_counts().sort_index()
credit_limit_group_default_rate = credit_df.groupby("credit_limit_group", observed=False)["default_flag"].mean()

credit_limit_group_summary = pd.DataFrame({
    "客户人数": credit_limit_group_counts,
    "下一期违约率": credit_limit_group_default_rate
})
credit_limit_group_summary

In [ ]:
low_limit_default_rate = credit_limit_group_default_rate["低额度"]
high_limit_default_rate = credit_limit_group_default_rate["高额度"]

print("低额度客户违约率是高额度客户的", (low_limit_default_rate / high_limit_default_rate).round(2), "倍")
print("两组违约率相差", ((low_limit_default_rate - high_limit_default_rate) * 100).round(2), "个百分点")

**分析结论：** 违约率随授信额度提高整体呈下降趋势。低额度组违约率约为 **31.79%**，高额度组约为 **13.26%**，相差约 **18.53 个百分点**。该结果体现的是统计关联，不代表低授信额度本身导致违约。


### 4.2 年龄与违约风险

**分析问题：** 不同年龄客群的下一期违约风险是否存在结构性差异？

按照年龄区间分组，并比较各年龄组的下一期违约率。


In [ ]:
credit_df["age"].describe()

In [ ]:
credit_df["age_group"] = pd.cut(
    credit_df["age"],
    bins=[0, 25, 35, 45, 60, 80],
    labels=["25岁以下", "25-34岁", "35-44岁", "45-59岁", "60岁及以上"],
    right=False
)

age_group_counts = credit_df["age_group"].value_counts().sort_index()
age_group_default_rate = credit_df.groupby("age_group", observed=False)["default_flag"].mean()

age_group_summary = pd.DataFrame({
    "客户人数": age_group_counts,
    "下一期违约率": age_group_default_rate
})
age_group_summary

**分析结论：** 年龄与违约率并非简单线性关系，整体呈现“两端较高、中间较低”的特征。25岁以下和60岁及以上客群违约率相对较高，中间年龄客群相对较低，因此年龄更适合作为辅助风险特征，而非单独判断依据。


### 4.3 信用额度使用率与违约风险

**分析问题：** 额度使用压力较高的客户，是否表现出更高的后续违约风险？

使用 **9月账单金额 ÷ 授信额度** 构造9月额度使用率代理指标。对于负账单金额，将计算后的负使用率压到0；大于1的情况保留，用于识别超额使用客户。


In [ ]:
print(credit_df["bill_amount_sep"].describe())
print("账单金额小于0的客户：", (credit_df["bill_amount_sep"] < 0).sum())
print("账单金额超过授信额度的客户：", (credit_df["bill_amount_sep"] > credit_df["credit_limit"]).sum())

In [ ]:
credit_df["sep_credit_utilization_raw"] = credit_df["bill_amount_sep"] / credit_df["credit_limit"]
credit_df["sep_credit_utilization"] = credit_df["sep_credit_utilization_raw"].clip(lower=0)

credit_df["sep_credit_utilization"].describe()

In [ ]:
credit_df["sep_credit_utilization_group"] = pd.cut(
    credit_df["sep_credit_utilization"],
    bins=[0, 0.2, 0.5, 0.8, 1.0, float("inf")],
    labels=["低使用", "中低使用", "中高使用", "高使用", "超额使用"],
    right=False
)

credit_utilization_group_counts = credit_df["sep_credit_utilization_group"].value_counts().sort_index()
credit_utilization_group_default_rate = credit_df.groupby("sep_credit_utilization_group", observed=False)["default_flag"].mean()

credit_utilization_summary = pd.DataFrame({
    "客户人数": credit_utilization_group_counts,
    "下一期违约率": credit_utilization_group_default_rate
})
credit_utilization_summary

In [ ]:
over_limit_default_rate = credit_utilization_group_default_rate["超额使用"]
low_utilization_default_rate = credit_utilization_group_default_rate["低使用"]

print("超额使用客户违约率是低使用客户的", (over_limit_default_rate / low_utilization_default_rate).round(2), "倍")
print("两组违约率相差", ((over_limit_default_rate - low_utilization_default_rate) * 100).round(2), "个百分点")

**分析结论：** 额度使用率与下一期违约风险总体呈正向关系，但并非严格单调。超额使用组违约率约为 **30.05%**，低使用组约为 **18.30%**。较高额度使用压力可作为补充风险信号，但不能单独作为违约判断依据。


### 4.4 还款覆盖情况与违约风险

**分析问题：** 当前还款金额相对账单金额较低的客户，是否具有更高的后续违约风险？

使用 **9月还款金额 ÷ 9月账单金额** 构造还款覆盖比例，仅在账单金额 `> 0` 时计算，以避免0或负账单造成无意义的比例。


In [ ]:
print(credit_df["payment_amount_sep"].describe())
print("账单金额等于0的客户：", (credit_df["bill_amount_sep"] == 0).sum())
print("账单金额小于等于0的客户：", (credit_df["bill_amount_sep"] <= 0).sum())

In [ ]:
positive_bill_mask = credit_df["bill_amount_sep"] > 0

positive_bill_amount_sep = credit_df.loc[positive_bill_mask, "bill_amount_sep"]
payment_amount_sep_for_positive_bill = credit_df.loc[positive_bill_mask, "payment_amount_sep"]

credit_df["sep_payment_to_bill_ratio"] = payment_amount_sep_for_positive_bill / positive_bill_amount_sep

credit_df["sep_payment_to_bill_ratio"].describe()

In [ ]:
print("无法计算还款比例的客户：", credit_df["sep_payment_to_bill_ratio"].isna().sum())
print("还款比例等于0的客户：", (credit_df["sep_payment_to_bill_ratio"] == 0).sum())
print("还款比例大于1的客户：", (credit_df["sep_payment_to_bill_ratio"] > 1).sum())
print("还款比例大于10的客户：", (credit_df["sep_payment_to_bill_ratio"] > 10).sum())

In [ ]:
credit_df["sep_payment_to_bill_group"] = pd.cut(
    credit_df["sep_payment_to_bill_ratio"],
    bins=[-1, 0, 0.05, 0.10, 0.30, 1, float("inf")],
    labels=["零还款", "极低还款", "较低还款", "中等还款", "较高还款", "超额还款"]
)

payment_to_bill_group_counts = credit_df["sep_payment_to_bill_group"].value_counts().sort_index()
payment_to_bill_group_default_rate = credit_df.groupby("sep_payment_to_bill_group", observed=False)["default_flag"].mean()

payment_to_bill_summary = pd.DataFrame({
    "客户人数": payment_to_bill_group_counts,
    "下一期违约率": payment_to_bill_group_default_rate
})
payment_to_bill_summary

In [ ]:
zero_payment_default_rate = payment_to_bill_group_default_rate["零还款"]
overpayment_default_rate = payment_to_bill_group_default_rate["超额还款"]

print("零还款客户违约率是超额还款客户的", (zero_payment_default_rate / overpayment_default_rate).round(2), "倍")
print("两组违约率相差", ((zero_payment_default_rate - overpayment_default_rate) * 100).round(2), "个百分点")

**分析结论：** 零还款组违约率约为 **39.77%**，超额还款组约为 **13.81%**，说明还款覆盖能力较弱的客群表现出更高的后续违约风险。由于公开数据的账单与还款字段存在口径限制，该指标仅作为探索性风险代理变量，不等同于严格业务口径下的“还款率”。


### 4.5 核心衍生风险特征

| 衍生变量 | 含义 |
|---|---|
| `overdue_month_count_6m` | 近6个月发生正向逾期的月份数 |
| `max_overdue_months_6m` | 近6个月最大正向逾期程度 |
| `continuous_overdue_flag_6m` | 近6个月是否存在相邻月份连续逾期 |
| `sep_credit_utilization` | 9月账单金额 / 授信额度 |
| `sep_payment_to_bill_ratio` | 9月还款金额 / 9月账单金额（仅正账单） |

这些衍生变量将作为后续综合风险规则的基础。


## 5. 综合风险规则与客户分层

在单特征分析基础上，选取5项具有业务可解释性的风险信号：

1. 近6个月逾期月份数 ≥ 3；
2. 近6个月最大逾期程度 ≥ 2；
3. 近6个月存在连续逾期；
4. 9月信用额度使用率 ≥ 80%；
5. 9月账单金额大于0且还款金额为0。

将每个风险信号统一表示为 True / False，并统计每位客户的规则命中数量，以观察多风险信号叠加后的违约率变化。

> 这些阈值基于当前数据集的探索结果设置，仅用于展示风险规则构建和分层思路，不代表真实业务中的正式审批、拦截或授信策略。


In [ ]:
credit_df["frequent_overdue_flag_6m"] = credit_df["overdue_month_count_6m"] >= 3
credit_df["severe_overdue_flag_6m"] = credit_df["max_overdue_months_6m"] >= 2
credit_df["high_sep_credit_utilization_flag"] = credit_df["sep_credit_utilization"] >= 0.8
credit_df["zero_sep_payment_flag"] = (credit_df["bill_amount_sep"] > 0) & (credit_df["sep_payment_to_bill_ratio"] == 0)

print("频繁逾期：", credit_df["frequent_overdue_flag_6m"].sum())
print("严重逾期：", credit_df["severe_overdue_flag_6m"].sum())
print("连续逾期：", credit_df["continuous_overdue_flag_6m"].sum())
print("高额度使用：", credit_df["high_sep_credit_utilization_flag"].sum())
print("零还款：", credit_df["zero_sep_payment_flag"].sum())

In [ ]:
risk_rule_columns = [
    "frequent_overdue_flag_6m",
    "severe_overdue_flag_6m",
    "continuous_overdue_flag_6m",
    "high_sep_credit_utilization_flag",
    "zero_sep_payment_flag"
]

credit_df["risk_rule_count"] = credit_df[risk_rule_columns].sum(axis=1)

risk_rule_customer_count = credit_df["risk_rule_count"].value_counts().sort_index()
risk_rule_default_rate = credit_df.groupby("risk_rule_count")["default_flag"].mean()

risk_rule_summary = pd.DataFrame({
    "客户人数": risk_rule_customer_count,
    "下一期违约率": risk_rule_default_rate
})
risk_rule_summary

**规则命中结果：** 随着风险规则命中数量增加，下一期违约率明显上升。命中0条规则的客户违约率约为 **11.95%**，命中3条规则约为 **53.91%**，命中5条规则约为 **57.85%**。高命中数组别样本量相对较小，因此需要结合样本规模谨慎解释。


In [ ]:
credit_df["risk_level"] = pd.cut(
    credit_df["risk_rule_count"],
    bins=[-1, 0, 2, 5],
    labels=["低风险", "中风险", "高风险"],
    right=True
)

risk_level_customer_count = credit_df["risk_level"].value_counts().sort_index()
risk_level_default_rate = credit_df.groupby("risk_level", observed=False)["default_flag"].mean()

risk_level_summary = pd.DataFrame({
    "客户人数": risk_level_customer_count,
    "下一期违约率": risk_level_default_rate
})
risk_level_summary

### 风险分层结果

- **低风险（命中0条）：** 违约率约 **11.95%**
- **中风险（命中1-2条）：** 违约率约 **21.56%**
- **高风险（命中3-5条）：** 违约率约 **55.88%**

风险等级从低到高呈现出明显的违约率梯度，说明多个风险信号组合后能够进一步增强客群风险区分度。该分层属于探索性规则分层，仍需通过独立样本和业务数据进一步验证。


## 6. 核心发现与风险启示

1. **历史逾期行为是本项目中最明显的风险信号。** 最近一期逾期、逾期频率、最大逾期程度和连续逾期均与下一期违约率存在明显关联。
2. **账户压力特征能够提供补充信息。** 低授信额度、高额度使用率和零还款客群均表现出较高的违约率，可与逾期类指标结合使用。
3. **多风险信号叠加后区分度进一步增强。** 规则命中数量增加时，下一期违约率整体明显上升；低、中、高风险层级呈现清晰的风险梯度。
4. **风险变量应结合使用，而非单一决策。** 年龄、额度使用率等单变量存在非线性或非严格单调关系，更适合作为组合风险判断的一部分。
5. **当前结果适合用于风险识别与策略探索。** 若进一步用于真实业务，还需要进行样本外验证、阈值校准，并结合误伤率、通过率等业务指标评估策略效果。


## 7. 项目局限性

- 数据为公开历史数据，字段和业务口径有限，无法完整还原真实金融机构的授信、账户管理和贷后场景；
- 当前分析主要采用单变量分组和可解释规则组合，没有进行训练集/测试集划分，也未构建机器学习模型；
- 风险规则阈值主要依据本数据集的探索结果设置，不能直接迁移为真实业务中的审批、授信或拦截阈值；
- 部分高逾期等级和高规则命中数组别样本量较小，需要结合样本规模谨慎解释；
- 本项目分析的是统计关联和风险区分能力，不用于证明变量与违约之间的因果关系。

**项目定位：** 消费信贷违约风险特征分析与可解释风险分层项目，重点展示 Python / Pandas 数据分析、风险特征工程、规则构建和业务解释能力。
